# SmolVLA fine-tune on SO-101 (PCB pick-and-place) — Google Colab

Cloud-training companion to the `smolvla_so101/` guide. This notebook:
1. Authenticates with Hugging Face
2. Pulls your **dataset** and the **`lerobot/smolvla_base`** model from the Hub
3. Fine-tunes SmolVLA on a cloud GPU
4. Uploads the fine-tuned checkpoint back to the Hub so you can download it and deploy locally

**Before running:** `Runtime → Change runtime type → GPU` (A100 or L4 recommended; T4 works with a smaller batch).


## 0. Check the GPU

In [1]:
!nvidia-smi

Wed Jul 22 19:51:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             45W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 1. Install LeRobot (with the SmolVLA extra)

Pinned to match your local `lerobot 0.5.2`. If pip reports dependency conflicts, run this cell, then
**Runtime → Restart session**, and continue from step 2 (do **not** re-run this cell after restart).


In [2]:
!pip install -q "lerobot[smolvla]==0.6.0"

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 97.8 MB/s eta 0:00:00


In [3]:
!pip install 'lerobot[dataset]'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.9/39.9 MB 45.2 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


## 2. Authenticate with Hugging Face

Recommended: add a **write** token to Colab Secrets (the 🔑 icon in the left sidebar) named `HF_TOKEN`,
then run this cell. Otherwise it falls back to an interactive login prompt. A **write** token is required
to upload the fine-tuned model in step 6.


In [4]:
from huggingface_hub import login
try:
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))
    print("Logged in via Colab secret HF_TOKEN")
except Exception:
    print("No HF_TOKEN secret found - falling back to interactive login...")
    login()

Logged in via Colab secret HF_TOKEN


In [5]:
!hf auth whoami

Hint: A new version of huggingface_hub (1.24.0) is available! You are using version 1.23.0.
To update, run: hf update
✓ Logged in
  user: HALDijkstraaa


## 3. Weights & Biases (W&B) login

Enables the live training dashboard (loss curves, learning rate, grad norm). Add a **`WANDB_API_KEY`** to
Colab Secrets (get it from https://wandb.ai/authorize) — otherwise this falls back to an interactive
prompt. To skip W&B entirely, set `USE_WANDB = False` in the next cell (then you can skip this login).


In [6]:
!pip install wandb

In [7]:
import wandb
try:
    from google.colab import userdata
    wandb.login(key=userdata.get("WANDB_API_KEY"))
    print("Logged into W&B via Colab secret WANDB_API_KEY")
except Exception:
    print("No WANDB_API_KEY secret found - falling back to interactive login...")
    wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: bj-huang (bj-huang-university-of-toronto) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Logged into W&B via Colab secret WANDB_API_KEY


## 4. Configuration — set your repo ids and hyperparameters

Set `DATASET_REPO_ID` to the dataset you pushed in `03_collect_data.md`. **Check your Hub profile for the
exact name** — the recorded dataset may carry a timestamp (e.g. `so101_pick_place_pcb_20260721_183543`).
The dataset must already be on the Hub (you recorded with `--dataset.push_to_hub=true`). `MODEL_REPO_ID`
is the new repo this notebook creates for the fine-tuned model.


In [8]:
HF_USER         = "HALDijkstraaa"
DATASET_REPO_ID = f"{HF_USER}/so101_pick_place_pcb_20260721_183543"         # <-- your dataset on the Hub (check exact name!)
BASE_MODEL      = "lerobot/smolvla_base"                     # forward slash is fine on Linux/Colab
MODEL_REPO_ID   = f"{HF_USER}/smolvla_so101_pcb_finetuned_v2"   # <-- output model repo (created in step 6)
OUTPUT_DIR      = "outputs/train/smolvla_so101_pcb_v2"

BATCH_SIZE = 64       # A100/L4: 8 is safe. Drop to 4 or 2 on a T4 if you hit OOM.
STEPS      = 20000   # a real fine-tune (~a few hours on A100). Lower (e.g. 2000) for a quick smoke test.
SAVE_FREQ  = 2000    # checkpoint every N steps (and always at the final step)

USE_WANDB     = True                 # set False to train without W&B logging
WANDB_PROJECT = "smolvla_so101_pcb_v2"  # your W&B project name (created on first run)

print("Dataset   :", DATASET_REPO_ID)
print("Base model:", BASE_MODEL)
print("Output ->  :", MODEL_REPO_ID)
print("W&B       :", f"{WANDB_PROJECT} (enabled={USE_WANDB})")

Dataset   : HALDijkstraaa/so101_pick_place_pcb_20260721_183543
Base model: lerobot/smolvla_base
Output ->  : HALDijkstraaa/smolvla_so101_pcb_finetuned_v2
W&B       : smolvla_so101_pcb_v2 (enabled=True)


## 5. Fine-tune

Uses `--policy.path` (a plain-string repo id — this is the flag that avoids the Windows `Path`→backslash
bug you hit locally). The dataset is pulled from the Hub automatically, so **no `--dataset.root`** is
needed in the cloud. Training output streams live below; with `USE_WANDB=True` a **W&B run URL** prints
near the top — open it for live loss / lr / grad-norm curves. `--wandb.disable_artifact=true` keeps W&B
from also uploading the (large) checkpoint, since you push it to the Hub in step 7.


In [ ]:
TRAIN_CMD = (
    "lerobot-train"
    f" --dataset.repo_id={DATASET_REPO_ID}"
    f" --policy.path={BASE_MODEL}"
    f" --output_dir={OUTPUT_DIR}"
    " --job_name=smolvla_so101_pcb"
    " --policy.device=cuda"
    f" --batch_size={BATCH_SIZE}"
    f" --steps={STEPS}"
    f" --save_freq={SAVE_FREQ}"
    " --log_freq=100"
    " --policy.push_to_hub=false"
    f" --wandb.enable={'true' if USE_WANDB else 'false'}"
    f" --wandb.project={WANDB_PROJECT}"
    " --wandb.disable_artifact=true"
    " --rename_map='{\"observation.images.front\": \"observation.images.camera1\"}'"
)
print(TRAIN_CMD)
!{TRAIN_CMD}

lerobot-train --dataset.repo_id=HALDijkstraaa/so101_pick_place_pcb_20260721_183543 --policy.path=lerobot/smolvla_base --output_dir=outputs/train/smolvla_so101_pcb_v2 --job_name=smolvla_so101_pcb --policy.device=cuda --batch_size=64 --steps=20000 --save_freq=2000 --log_freq=100 --policy.push_to_hub=false --wandb.enable=true --wandb.project=smolvla_so101_pcb_v2 --wandb.disable_artifact=true --rename_map='{"observation.images.front": "observation.images.camera1"}'
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
config.json: 100% 2.30k/2.30k [00:00<00:00, 4.31MB/s]
INFO 2026-07-22 19:56:07 ot_train.py:232 {'batch_size': 64,
 'checkpoint_path': None,
 'cudnn_deterministic': False,
 'dataset': {'depth_output_unit': 'mm',
             'episodes'

## 6. Locate the fine-tuned checkpoint

lerobot saves to `OUTPUT_DIR/checkpoints/<step>/pretrained_model/` — weights `model.safetensors` plus
`config.json` and `train_config.json`. On Linux the `last` symlink is created fine, so we prefer it.


In [ ]:
import os, glob
last = os.path.join(OUTPUT_DIR, "checkpoints", "last", "pretrained_model")
if os.path.exists(last):
    CKPT_DIR = last
else:
    CKPT_DIR = sorted(glob.glob(os.path.join(OUTPUT_DIR, "checkpoints", "*", "pretrained_model")))[-1]
print("Checkpoint dir:", CKPT_DIR)
print("Contents:", os.listdir(CKPT_DIR))

IndexError: list index out of range

## 7. Upload the fine-tuned model to the Hub

Creates a **private** model repo and uploads the checkpoint's `pretrained_model/` folder (config + weights).


In [ ]:
from huggingface_hub import HfApi
api = HfApi()
api.create_repo(MODEL_REPO_ID, repo_type="model", private=True, exist_ok=True)
api.upload_folder(
    folder_path=CKPT_DIR,
    repo_id=MODEL_REPO_ID,
    repo_type="model",
    commit_message="Fine-tuned SmolVLA on SO-101 PCB pick-and-place",
)
print(f"Uploaded -> https://huggingface.co/{MODEL_REPO_ID}")

## 8. Download later (on your local machine) and deploy

Back on your Windows box (in the `lerobot` conda env), pull the fine-tuned model into a **local folder**
and point the deploy command at that folder:

```powershell
huggingface-cli download HALDijkstraaa/smolvla_so101_pcb_finetuned --local-dir D:\models\smolvla_so101_pcb_finetuned
```

Then deploy with `--policy.path="D:\models\smolvla_so101_pcb_finetuned"` (see `04_finetune_and_deploy.md`).
Pointing at a local folder also sidesteps the Windows Hub-repo-id backslash issue.
